In [68]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

# importerer data
stations_df = pd.read_csv("../raw_data/stations.csv")
trips_df = pd.read_csv("../raw_data/trips.csv")
weather_df = pd.read_csv("../raw_data/weather.csv")

In [69]:
# korrigerer til riktig tidssone før videre analyse
# PS - vil endre dtype på timestamp til Europe/Oslo
stations_df["timestamp"] = pd.to_datetime(stations_df["timestamp"]).dt.tz_convert("Europe/Oslo")
trips_df["started_at"] = pd.to_datetime(trips_df["started_at"],format="ISO8601").dt.tz_convert("Europe/Oslo")
trips_df["ended_at"] = pd.to_datetime(trips_df["ended_at"],format="ISO8601").dt.tz_convert("Europe/Oslo")
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"]).dt.tz_convert("Europe/Oslo")

In [70]:
# filtrerer data i trips og weather til å kun omhandle perioden stations måler
start_time = stations_df["timestamp"].min()
end_time = stations_df["timestamp"].max()

trips_df = trips_df[(trips_df["started_at"] >= start_time) & (trips_df["started_at"] <= end_time)]
trips_df = trips_df[(trips_df["ended_at"] >= start_time) & (trips_df["ended_at"] <= end_time)]
weather_df = weather_df[(weather_df["timestamp"] >= start_time) & (weather_df["timestamp"] <= end_time)]

print("Målinger for stations startet", start_time, "og sluttet", end_time)
print("Målinger for oppdatert starttid i  trips startet", trips_df["started_at"].min(), 
      "og sluttet", trips_df["started_at"].max())
print("Målinger for oppdatert slutttid i  trips startet", trips_df["ended_at"].min(), 
      "og sluttet", trips_df["ended_at"].max())
print("Målinger for oppdatert weather started", weather_df["timestamp"].min(),
      "og sluttet", weather_df["timestamp"].max())

Målinger for stations startet 2024-04-10 23:45:34+02:00 og sluttet 2025-05-02 17:49:25+02:00
Målinger for oppdatert starttid i  trips startet 2024-04-11 05:28:38.527000+02:00 og sluttet 2025-05-02 17:44:09.545000+02:00
Målinger for oppdatert slutttid i  trips startet 2024-04-11 05:32:40.747000+02:00 og sluttet 2025-05-02 17:48:40.297000+02:00
Målinger for oppdatert weather started 2024-04-11 00:00:00+02:00 og sluttet 2025-05-02 17:00:00+02:00


In [71]:
# preprosesserer csv-ene til felles analyse

# OpenAI's ChatGPT ble brukt til å formulere og/eller komme opp med ideer for deler av kode.
# Dette gjelder target_stations_df, der det blri brukt resampling ihht til LCOF-prinsippet.
# All kode som er hjulpet av AI er forstått i seg selv og ihht til resten av programmet.
# OpenAI - https://chatgpt.com - henta 09.09.25 

# preprosseserer stations_df
target_stations = np.array(["Møllendalsplass", "Torgallmenningen", "Grieghallen",
                            "Høyteknologisenteret", "Studentboligene", "Akvariet",
                            "Damsgårdsveien 71", "Dreggsallmenningen Sør", "Florida Bybanestopp"])

filtered_stations = stations_df[stations_df["station"].isin(target_stations)]
filtered_stations = filtered_stations.set_index("timestamp").sort_index()
target_stations_df = filtered_stations.groupby("station")["free_bikes"].apply(lambda s: s.resample("1h").ffill()
                                        ).reset_index().pivot(index="timestamp", columns="station", values="free_bikes").sort_index()

target_stations_df.columns.name = None
target_stations_df = target_stations_df.reset_index()

# preprosseserer trips_df
trips_df["duration_min"] = (trips_df["ended_at"] - trips_df["started_at"]).dt.total_seconds() / 60
trips_hourly = trips_df.groupby(trips_df["started_at"].dt.floor("h")).size().reset_index(name="trips_count")
trips_hourly = trips_hourly.rename(columns={"started_at": "timestamp"})
trips_hourly_stations = trips_df.groupby([trips_df["started_at"].dt.floor("h"), "start_station_name"]).size().reset_index(name="trips_count")
trips_hourly_stations = trips_hourly_stations.rename(columns={"started_at": "timestamp"})

# slår sammen alt til to dataframes
# merged_df har alle stasjoner, sykkelbruk og vær
# merged_stations_df har turer per startstasjon og vær
merged_df = target_stations_df.merge(trips_hourly, on="timestamp", how="left")
merged_df = merged_df.merge(weather_df, on="timestamp", how="left")
merged_stations_df = trips_hourly_stations.merge(weather_df, on="timestamp", how="left")

print(merged_df.info())
print(merged_df.describe())
print(merged_stations_df.info())
print(merged_stations_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9284 entries, 0 to 9283
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype                      
---  ------                  --------------  -----                      
 0   timestamp               9284 non-null   datetime64[ns, Europe/Oslo]
 1   Akvariet                9282 non-null   float64                    
 2   Damsgårdsveien 71       9283 non-null   float64                    
 3   Dreggsallmenningen Sør  9283 non-null   float64                    
 4   Florida Bybanestopp     9283 non-null   float64                    
 5   Grieghallen             9283 non-null   float64                    
 6   Høyteknologisenteret    9283 non-null   float64                    
 7   Møllendalsplass         9283 non-null   float64                    
 8   Studentboligene         9283 non-null   float64                    
 9   Torgallmenningen        9283 non-null   float64                    
 10  trips_count 

In [72]:

merged_df.head()

,timestamp,Akvariet,Damsgårdsveien 71,Dreggsallmenningen Sør,Florida Bybanestopp,Grieghallen,Høyteknologisenteret,Møllendalsplass,Studentboligene,Torgallmenningen,trips_count,temperature,precipitation,wind_speed
0,2024-04-10 23:00:00+02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-11 00:00:00+02:00,0.0,12.0,12.0,0.0,14.0,0.0,4.0,15.0,15.0,NaN,5.1,0.2,18.4
2,2024-04-11 01:00:00+02:00,0.0,12.0,12.0,0.0,14.0,0.0,4.0,16.0,15.0,NaN,5.0,0.3,19.0
3,2024-04-11 02:00:00+02:00,0.0,12.0,12.0,0.0,14.0,0.0,4.0,16.0,15.0,NaN,5.2,0.3,19.1
4,2024-04-11 03:00:00+02:00,0.0,12.0,12.0,0.0,14.0,0.0,4.0,16.0,15.0,NaN,5.3,0.8,21.8


In [73]:
merged_stations_df.head()

,timestamp,start_station_name,trips_count,temperature,precipitation,wind_speed
0,2024-04-11 05:00:00+02:00,Gågaten,1,5.5,1.3,24.0
1,2024-04-11 05:00:00+02:00,Torgallmenningen,1,5.5,1.3,24.0
2,2024-04-11 06:00:00+02:00,Busstasjonen 2 Vest,1,5.8,2.2,24.0
3,2024-04-11 06:00:00+02:00,C. Sundts gate,1,5.8,2.2,24.0
4,2024-04-11 06:00:00+02:00,Damsgårdsveien 2,1,5.8,2.2,24.0


In [74]:
# lager korrelasjonsmatrise mellom deler i merged_df
corr = merged_df.drop(columns="timestamp").corr(numeric_only=True)
fig = px.imshow(corr, text_auto=True, title="Korrelasjonsmatrise")
fig.show()

In [75]:
# En enkel scatter: antall turer vs. ledige sykler på Grieghallen-stasjon via scatter
fig = px.scatter(merged_df, "trips_count", "Grieghallen",
                 color="temperature", title="Turer vs. ledige sykler på Grieghallen-stasjon")
fig.update_layout()
fig.show()

In [76]:
# visualiserer sykkelbruk og temperatur over tid via lineplot
fig = px.line(merged_df, x="timestamp", y=["trips_count", "temperature"],
              title="Sykkelbruk og temperatur over tid")
fig.update_layout(yaxis_title="antall turer / temperatur")
fig.show()

In [77]:
# visualiserer sammenhengen mellom temperatur og ledige sykler på Grieghallen-stasjon via lineplot
fig = px.line(merged_df, "timestamp", ["temperature", "Grieghallen"],
              title="temperatur vs ledige sykler på Grieghallen-stasjon")
fig.update_layout(yaxis_title="ledige sykler / temperatur")
fig.show()

In [78]:
# visualiserer sammenheng mellom sykkelbruk og temperatur via scatter
fig = px.scatter(merged_df, "temperature", "trips_count", title="Sykkelbruk vs temperatur")
fig.update_layout(yaxis_title="antall turer")
fig.show()

In [79]:
# visualiserer sammenheng mellom sykkelbruk og vind via scatter
fig = px.scatter(merged_df, x="wind_speed", y="trips_count", title="Sykkelbruk vs vind")
fig.update_layout(yaxis_title="antall turer")
fig.show()

In [80]:
# visualiserer sammenheng mellom sykkelbruk og nedbør via scatter
fig = px.scatter(merged_df, x="precipitation", y="trips_count", title="Sykkelbruk vs. nedbør")
fig.update_layout(yaxis_title="antall turer")
fig.show()